# ACOUSLIC-AI Attention U-Net — subject-level retrain + one-time held-out test

**Purpose.** The original `aa-unet-training` run used a case-level 85/15 split: the 45 validation cases picked the checkpoint and the frame-selection rule, and 7 of them came from women also in training. This notebook gives a clean, citable result:

* split **by woman (`subject_id`)** into train 70% / validation 15% / test 15% (seed 42);
* validation is used only to choose the checkpoint (best Dice);
* the AC pipeline is **frozen before training** (top-5 most confident frames per sweep, median AC, min. component 2,000 px, threshold 0.5);
* the test set is scored **once**, at the end, against the expert per-sweep reference AC.

**Settings:** Accelerator = GPU (T4 or P100), Internet = ON. Attach the dataset `patelvru/acouslic-aidataset`. Run all cells in order (about 2–3 hours, mostly training). If the session stops, re-run cells 1–6, then cell 7 resumes from its checkpoint.

In [ ]:
# Cell 1 — setup
import os, glob, random, math, time, json, zlib
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
try:
    import SimpleITK as sitk
except ImportError:
    os.system("pip install -q SimpleITK"); import SimpleITK as sitk

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
H, W = 320, 416; ORIG_H, ORIG_W = 562, 744
FRAMES_PER_SWEEP = 140; N_SWEEPS = 6
EMPTY_RATIO = 0.2
EPOCHS, BS, LR = 50, 16, 3e-4
# Frozen AC pipeline (fixed BEFORE looking at the test set)
MIN_AREA, THRESH, TOP_K = 2000, 0.5, 5
OUT = Path("/kaggle/working"); PROC = OUT / "proc"; PROC.mkdir(exist_ok=True, parents=True)
dev = torch.device("cuda"); print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 — pair images, masks and the reference-AC file
all_mha = sorted(glob.glob("/kaggle/input/**/*.mha", recursive=True))
msk_files = {Path(p).stem: p for p in all_mha if "mask" in p.lower()}
img_files = {Path(p).stem: p for p in all_mha if "mask" not in p.lower()}
case_ids = sorted(set(img_files) & set(msk_files))
CSV_PATH = next(c for c in glob.glob("/kaggle/input/**/*.csv", recursive=True) if "circ" in c.lower())
ref = pd.read_csv(CSV_PATH, dtype={"uuid": str, "subject_id": str})
SWEEP_COLS = [f"sweep_{k}_ac_mm" for k in range(1, 7)]
subject_of = ref.set_index("uuid")["subject_id"].to_dict()
print("paired cases:", len(case_ids), "| subjects:", ref.subject_id.nunique(), "| CSV:", CSV_PATH)
assert len(case_ids) == 300 and all(c in subject_of for c in case_ids)

In [ ]:
# Cell 3 — SUBJECT-LEVEL split 70 / 15 / 15 (no woman appears in two sets)
subjects = sorted(set(subject_of[c] for c in case_ids))
rng = random.Random(SEED); rng.shuffle(subjects)
n_test = int(round(0.15 * len(subjects))); n_val = int(round(0.15 * len(subjects)))
test_s, val_s = set(subjects[:n_test]), set(subjects[n_test:n_test + n_val])
train_s = set(subjects[n_test + n_val:])
split = {c: ("test" if subject_of[c] in test_s else "val" if subject_of[c] in val_s else "train") for c in case_ids}
train_ids = [c for c in case_ids if split[c] == "train"]
val_ids   = [c for c in case_ids if split[c] == "val"]
test_ids  = [c for c in case_ids if split[c] == "test"]
assert not (train_s & val_s or train_s & test_s or val_s & test_s)
for name, ids in [("train", train_ids), ("val", val_ids), ("test", test_ids)]:
    (OUT / f"split_{name}_uuids.txt").write_text("\n".join(ids) + "\n")
    n_ref = int(ref.set_index("uuid").loc[ids, SWEEP_COLS].notna().sum().sum())
    print(f"{name:5s}: {len(ids):3d} cases | {len({subject_of[c] for c in ids}):3d} women | {n_ref} reference sweeps")

In [ ]:
# Cell 4 — preprocess every case once (annotated frames + 20% empty frames, resized)
def preprocess_case(cid):
    out = PROC / f"{cid}.npz"
    if out.exists(): return
    img = sitk.GetArrayFromImage(sitk.ReadImage(img_files[cid]))
    msk = sitk.GetArrayFromImage(sitk.ReadImage(msk_files[cid])).astype(np.uint8)
    if img.dtype != np.uint8:
        img = (255 * img.astype(np.float32) / max(float(img.max()), 1)).astype(np.uint8)
    has = (msk.reshape(msk.shape[0], -1) > 0).any(1)
    pos, neg = np.where(has)[0], np.where(~has)[0]
    r = np.random.RandomState(zlib.crc32(cid.encode()))          # per-case, reproducible
    n_neg = min(len(neg), int(EMPTY_RATIO * len(pos)) + (2 if len(pos) == 0 else 0))
    neg = r.choice(neg, n_neg, replace=False) if n_neg > 0 else np.array([], int)
    idx = np.sort(np.concatenate([pos, neg]).astype(int))
    X = np.stack([cv2.resize(img[i], (W, H), interpolation=cv2.INTER_AREA) for i in idx]) if len(idx) else np.zeros((0, H, W), np.uint8)
    Y = np.stack([cv2.resize(msk[i], (W, H), interpolation=cv2.INTER_NEAREST) for i in idx]) if len(idx) else np.zeros((0, H, W), np.uint8)
    np.savez_compressed(out, x=X, y=Y, fi=idx)

t0 = time.time()
for k, cid in enumerate(train_ids + val_ids):          # test cases are NOT preprocessed for training
    preprocess_case(cid)
    if k % 25 == 0: print(f"{k}/{len(train_ids) + len(val_ids)} cases, {time.time()-t0:.0f}s", flush=True)

def load_split(ids):
    X, Y = [], []
    for cid in ids:
        d = np.load(PROC / f"{cid}.npz")
        if len(d["x"]): X.append(d["x"]); Y.append(d["y"])
    return np.concatenate(X), np.concatenate(Y)
Xtr, Ytr = load_split(train_ids); Xva, Yva = load_split(val_ids)
print("train frames", len(Xtr), "| val frames", len(Xva))

In [ ]:
# Cell 5 — data pipeline (same augmentation as aa-unet-training)
def augment(x, y):
    if random.random() < 0.5: x, y = x[:, ::-1], y[:, ::-1]
    x, y = np.ascontiguousarray(x), np.ascontiguousarray(y)
    M = cv2.getRotationMatrix2D((W / 2, H / 2), random.uniform(-15, 15), random.uniform(0.9, 1.1))
    M[0, 2] += random.uniform(-0.05, 0.05) * W; M[1, 2] += random.uniform(-0.05, 0.05) * H
    x = cv2.warpAffine(x, M, (W, H), flags=cv2.INTER_LINEAR, borderValue=0)
    y = cv2.warpAffine(y, M, (W, H), flags=cv2.INTER_NEAREST, borderValue=0)
    x = x.astype(np.float32) / 255.0
    if random.random() < 0.5: x = np.clip(x, 0, 1) ** random.uniform(0.7, 1.4)
    if random.random() < 0.5: x = x * random.uniform(0.85, 1.15) + random.uniform(-0.08, 0.08)
    if random.random() < 0.3: x = x + np.random.randn(*x.shape).astype(np.float32) * random.uniform(0, 0.04)
    return np.clip(x, 0, 1).astype(np.float32), y

class FrameDS(Dataset):
    def __init__(s, X, Y, train): s.X, s.Y, s.train = X, Y, train
    def __len__(s): return len(s.X)
    def __getitem__(s, i):
        x, lab = s.X[i], s.Y[i]; y = (lab > 0).astype(np.uint8)
        if s.train: x, y = augment(x, y)
        else: x = x.astype(np.float32) / 255.0
        return torch.from_numpy(x)[None], torch.from_numpy(y.astype(np.float32))[None]

train_dl = DataLoader(FrameDS(Xtr, Ytr, True), batch_size=BS, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
val_dl = DataLoader(FrameDS(Xva, Yva, False), batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# Cell 6 — model (identical to best_attunet.pth architecture) and loss
class ConvBlock(nn.Module):
    def __init__(s, i, o):
        super().__init__()
        s.b = nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True),
                            nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True))
    def forward(s, x): return s.b(x)
class AttGate(nn.Module):
    def __init__(s, c, mid):
        super().__init__()
        s.Wg = nn.Sequential(nn.Conv2d(c, mid, 1, bias=False), nn.BatchNorm2d(mid))
        s.Wx = nn.Sequential(nn.Conv2d(c, mid, 1, bias=False), nn.BatchNorm2d(mid))
        s.psi = nn.Sequential(nn.Conv2d(mid, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(s, g, x): return x * s.psi(F.relu(s.Wg(g) + s.Wx(x)))
class AttUNet(nn.Module):
    def __init__(s, ch=(32, 64, 128, 256, 512)):
        super().__init__()
        s.enc, i = nn.ModuleList(), 1
        for c in ch: s.enc.append(ConvBlock(i, c)); i = c
        s.pool = nn.MaxPool2d(2)
        s.up, s.att, s.dec = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        for k in range(len(ch) - 1, 0, -1):
            s.up.append(nn.ConvTranspose2d(ch[k], ch[k-1], 2, stride=2))
            s.att.append(AttGate(ch[k-1], max(ch[k-1] // 2, 8)))
            s.dec.append(ConvBlock(ch[k-1] * 2, ch[k-1]))
        s.out = nn.Conv2d(ch[0], 1, 1)
    def forward(s, x):
        skips = []
        for k, e in enumerate(s.enc):
            x = e(x)
            if k < len(s.enc) - 1: skips.append(x); x = s.pool(x)
        for u, a, d in zip(s.up, s.att, s.dec):
            x = u(x); sk = a(x, skips.pop()); x = d(torch.cat([sk, x], 1))
        return s.out(x)

def loss_fn(logits, y):
    logits = logits.float(); p = torch.sigmoid(logits)
    return F.binary_cross_entropy_with_logits(logits, y) + 1 - (2 * (p * y).sum() + 1) / (p.sum() + y.sum() + 1)

@torch.no_grad()
def val_dice(model):
    model.eval(); ds = []
    for x, y in val_dl:
        with torch.autocast("cuda", dtype=torch.float16):
            pr = torch.sigmoid(model(x.to(dev)).float()) > THRESH
        pr, g = pr.cpu().numpy()[:, 0], y.numpy()[:, 0] > 0.5
        for p, gg in zip(pr, g):
            if gg.any(): ds.append(2 * (p & gg).sum() / (p.sum() + gg.sum() + 1e-6))
    return float(np.mean(ds))

In [ ]:
# Cell 7 — train; checkpoint = best VALIDATION Dice (test set untouched). Resumable.
CKPT, BEST = OUT / "ckpt_subject_split.pth", OUT / "best_attunet_subject_split.pth"
model = AttUNet().to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler("cuda"); best, start, log = -1.0, 1, []
if CKPT.exists():
    ck = torch.load(CKPT, map_location=dev)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
    scaler.load_state_dict(ck["scaler"]); best, start, log = ck["best"], ck["ep"] + 1, ck["log"]
    print("resumed at epoch", start)
for ep in range(start, EPOCHS + 1):
    model.train(); t0 = time.time(); run = 0
    for x, y in train_dl:
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16): lg = model(x)
        loss = loss_fn(lg, y); scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); run += loss.item()
    sched.step(); d = val_dice(model); log.append(dict(epoch=ep, loss=run / len(train_dl), val_dice=d))
    print(f"ep {ep:02d} loss {run/len(train_dl):.4f} | val Dice {d:.4f} | {time.time()-t0:.0f}s", flush=True)
    if d > best: best = d; torch.save(model.state_dict(), BEST)
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                    scaler=scaler.state_dict(), best=best, ep=ep, log=log), CKPT)
pd.DataFrame(log).to_csv(OUT / "training_log.csv", index=False); print("best val Dice:", best)

In [ ]:
# Cell 8 — the FROZEN AC pipeline (same as models/new_model/inference.py)
model = AttUNet().to(dev); model.load_state_dict(torch.load(BEST, map_location=dev)); model.eval()

def postprocess(p):
    p = p.astype(np.uint8); n, lab, st, _ = cv2.connectedComponentsWithStats(p, connectivity=8)
    if n <= 1: return np.zeros_like(p)
    out = (lab == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])).astype(np.uint8)
    return out if out.sum() >= MIN_AREA else np.zeros_like(p)

@torch.no_grad()
def predict_probs(frames, bs=32):
    out = []
    for i in range(0, len(frames), bs):
        x = torch.from_numpy(frames[i:i+bs].astype(np.float32) / 255)[:, None].to(dev)
        with torch.autocast("cuda", dtype=torch.float16): lg = model(x)
        out.append(torch.sigmoid(lg.float())[:, 0].cpu().numpy())
    return np.concatenate(out)

def ac_full(m, px):
    cnts, _ = cv2.findContours((m > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts: return np.nan
    c = max(cnts, key=cv2.contourArea)
    if len(c) < 5: return np.nan
    (_, _), (w, h), _ = cv2.fitEllipse(c.reshape(-1, 2).astype(np.float32))
    a, b = w / 2 * px, h / 2 * px
    return math.pi * (3 * (a + b) - math.sqrt((3 * a + b) * (a + 3 * b)))

def sweep_ac(frames, px):
    small = np.stack([cv2.resize(f, (W, H), interpolation=cv2.INTER_AREA) for f in frames])
    pr = predict_probs(small); pps = [postprocess(p > THRESH) for p in pr]
    area = np.array([m.sum() for m in pps], float)
    conf = np.array([p[m > 0].mean() if m.any() else 0 for p, m in zip(pr, pps)])
    valid = np.where(area > 0)[0]
    if len(valid) == 0: return np.nan, pr, pps
    best_i = valid[np.argsort(-conf[valid])[:TOP_K]]
    acs = [ac_full(cv2.resize(pr[i], (frames.shape[2], frames.shape[1]), interpolation=cv2.INTER_LINEAR) > THRESH, px) for i in best_i]
    return float(np.nanmedian(acs)), pr, pps
print("pipeline frozen: threshold", THRESH, "| min area", MIN_AREA, "| top-k", TOP_K)

In [ ]:
# Cell 9 — ONE-TIME TEST EVALUATION (run once; do not change anything above after seeing it)
reft = ref.set_index("uuid")
rows, dice_rows, t0 = [], [], time.time()
for n, cid in enumerate(test_ids):
    im = sitk.ReadImage(img_files[cid]); px = float(im.GetSpacing()[0])
    vol = sitk.GetArrayFromImage(im)
    if vol.dtype != np.uint8: vol = (255 * vol.astype(np.float32) / max(float(vol.max()), 1)).astype(np.uint8)
    msk = sitk.GetArrayFromImage(sitk.ReadImage(msk_files[cid]))
    for k in range(N_SWEEPS):
        s, e = k * FRAMES_PER_SWEEP, (k + 1) * FRAMES_PER_SWEEP
        ac, pr, pps = sweep_ac(vol[s:e], px)
        for j, m in enumerate(pps):                      # frame-level segmentation metrics
            g = cv2.resize(msk[s + j], (W, H), interpolation=cv2.INTER_NEAREST) > 0
            if g.any(): dice_rows.append(("annotated", 2 * (m.astype(bool) & g).sum() / (m.sum() + g.sum() + 1e-6)))
            else: dice_rows.append(("empty", float(m.any())))
        r = reft.loc[cid, SWEEP_COLS[k]]
        if pd.notna(r):
            rows.append(dict(uuid=cid, subject_id=subject_of[cid], sweep=k + 1, ref_ac_mm=float(r), pred_ac_mm=ac, px_mm=px))
    print(f"test case {n+1}/{len(test_ids)}, {time.time()-t0:.0f}s", flush=True)
pairs = pd.DataFrame(rows); pairs.to_csv(OUT / "test_ac_predictions_per_sweep.csv", index=False)
seg = pd.DataFrame(dice_rows, columns=["kind", "value"])

In [ ]:
# Cell 10 — agreement statistics (same definitions as fetal/evaluate_ac_accuracy.py)
def stats(r, p):
    d = p - r; sd = float(np.std(d, ddof=1))
    return dict(MAE_mm=float(np.mean(np.abs(d))), MedianAE_mm=float(np.median(np.abs(d))),
                RMSE_mm=float(np.sqrt(np.mean(d**2))), Bias_mm=float(np.mean(d)), SD_diff_mm=sd,
                LoA_lower_mm=float(np.mean(d) - 1.96*sd), LoA_upper_mm=float(np.mean(d) + 1.96*sd),
                Proportional_bias_slope=float(np.polyfit((p + r)/2, d, 1)[0]),
                NAE_mean=float(np.mean(np.abs(d) / np.maximum(np.maximum(r, p), 1e-6))),
                Within_8_8pct=float(np.mean(np.abs(d) / r * 100 <= 8.8)), Within_10mm=float(np.mean(np.abs(d) <= 10)))
det = pairs.dropna(subset=["pred_ac_mm"]).reset_index(drop=True)
m = stats(det.ref_ac_mm.values, det.pred_ac_mm.values)
groups = {s: g.index.values for s, g in det.groupby("subject_id")}; keys = list(groups)
rng = np.random.default_rng(42); boots = []
for _ in range(2000):
    idx = np.concatenate([groups[k] for k in rng.choice(keys, len(keys), replace=True)])
    if len(idx) > 2: boots.append(stats(det.ref_ac_mm.values[idx], det.pred_ac_mm.values[idx]))
B = pd.DataFrame(boots); ci = {c: [float(B[c].quantile(.025)), float(B[c].quantile(.975))] for c in B}
case = det.groupby(["uuid", "subject_id"])[["ref_ac_mm", "pred_ac_mm"]].mean()
mc = stats(case.ref_ac_mm.values, case.pred_ac_mm.values)
summary = dict(split="subject-level 70/15/15, seed 42", n_test_cases=len(test_ids),
               n_test_women=len({subject_of[c] for c in test_ids}), n_reference_sweeps=len(pairs),
               n_detection_failures=int(pairs.pred_ac_mm.isna().sum()), per_sweep=m, per_sweep_95CI=ci,
               per_case=mc, n_cases_scored=len(case),
               test_dice_annotated=float(seg[seg.kind == "annotated"].value.mean()),
               test_empty_frame_false_positive=float(seg[seg.kind == "empty"].value.mean()),
               best_val_dice=float(best))
(OUT / "test_ac_report.json").write_text(json.dumps(summary, indent=2))
f = lambda k, fmt="{:.1f}": f"{fmt.format(m[k])} ({fmt.format(ci[k][0])} to {fmt.format(ci[k][1])})"
txt = f"""ACOUSLIC-AI AC ACCURACY - INDEPENDENT HELD-OUT TEST (subject-level split, test used once)
Test: {summary['n_test_cases']} cases / {summary['n_test_women']} women / {summary['n_reference_sweeps']} reference sweeps
Detection failures: {summary['n_detection_failures']} ({summary['n_detection_failures']/max(len(pairs),1)*100:.1f}%)
Segmentation: Dice (annotated frames) {summary['test_dice_annotated']:.3f}; empty-frame false positives {summary['test_empty_frame_false_positive']*100:.1f}%
Per sweep (95% CI, cluster bootstrap over women):
- MAE {f('MAE_mm')} mm | median AE {m['MedianAE_mm']:.1f} mm | RMSE {f('RMSE_mm')} mm
- Bias (pred - ref) {f('Bias_mm', '{:+.1f}')} mm | SD {m['SD_diff_mm']:.1f} mm
- 95% limits of agreement {m['LoA_lower_mm']:+.1f} to {m['LoA_upper_mm']:+.1f} mm | proportional bias slope {m['Proportional_bias_slope']:+.3f}
- NAE {f('NAE_mean', '{:.3f}')} | within 8.8%: {m['Within_8_8pct']*100:.1f}% | within 10 mm: {m['Within_10mm']*100:.1f}%
Per case (n={len(case)}): MAE {mc['MAE_mm']:.1f} mm, bias {mc['Bias_mm']:+.1f} mm, LoA {mc['LoA_lower_mm']:+.1f} to {mc['LoA_upper_mm']:+.1f} mm
"""
(OUT / "test_ac_report.txt").write_text(txt); print(txt)

In [ ]:
# Cell 11 — Bland–Altman figure + list of files to download
import matplotlib; import matplotlib.pyplot as plt
r, p = det.ref_ac_mm.values, det.pred_ac_mm.values; mean, diff = (r + p) / 2, p - r
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
lim = (min(r.min(), p.min()) - 10, max(r.max(), p.max()) + 10)
ax[0].scatter(r, p, s=14, alpha=.7); ax[0].plot(lim, lim, "k--", lw=1)
ax[0].set(xlim=lim, ylim=lim, xlabel="Reference AC (mm)", ylabel="Predicted AC (mm)", title=f"Held-out test (n={len(det)} sweeps)")
ax[1].scatter(mean, diff, s=14, alpha=.7)
for y, ls in [(m["Bias_mm"], "-"), (m["LoA_upper_mm"], "--"), (m["LoA_lower_mm"], "--")]: ax[1].axhline(y, color="r", ls=ls)
ax[1].axhline(0, color="k", lw=.8); ax[1].set(xlabel="Mean of predicted and reference (mm)", ylabel="Predicted - reference (mm)", title="Bland-Altman")
plt.tight_layout(); plt.savefig(OUT / "test_bland_altman.png", dpi=200); plt.show()
for fn in ["best_attunet_subject_split.pth", "split_train_uuids.txt", "split_val_uuids.txt", "split_test_uuids.txt",
           "training_log.csv", "test_ac_predictions_per_sweep.csv", "test_ac_report.txt", "test_ac_report.json", "test_bland_altman.png"]:
    q = OUT / fn; print(f"{fn:38s}", f"{q.stat().st_size/1e6:.2f} MB" if q.exists() else "MISSING")